![](Images/banner_health_analytics.png){fig-align="center" width="100%"}

### Health Analytics with Python · Module 04: Machine Learning for Clinical Prediction
***
**Learning objectives**
- Understand why random splits fail in clinical settings and when temporal splits are required
- Build a reproducible preprocessing pipeline with `sklearn.Pipeline`
- Handle class imbalance with SMOTE, class weights, and under-sampling
- Apply stratified k-fold and time-series cross-validation correctly
- Establish a baseline model (majority classifier, logistic regression) to beat

**Estimated time:** 2 hours | **Level:** Intermediate | **Libraries:** `pandas`, `numpy`, `sklearn`, `imblearn`, `matplotlib`


## 1. Setup & Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      TimeSeriesSplit, cross_val_score)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              classification_report, confusion_matrix,
                              brier_score_loss, RocCurveDisplay)
import warnings
warnings.filterwarnings("ignore")
import os; os.makedirs("/tmp/mod04", exist_ok=True)

plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})

# ── Reconstruct shared dataset ────────────────────────────────────────────────
np.random.seed(42); N=3000
def sig(x): return 1/(1+np.exp(-x))
b = np.random.normal(size=(N,4)); np.random.seed(99)
age          = np.random.normal(62,15,N).clip(18,92).astype(int)
sex          = np.random.choice(["M","F"],N,p=[0.48,0.52])
payer        = np.random.choice(["Medicare","Medicaid","Private","Self-pay","Other"],N,p=[0.40,0.22,0.28,0.07,0.03])
admit_type   = np.random.choice(["Emergency","Elective","Urgent"],N,p=[0.52,0.30,0.18])
los_days     = np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)
diabetes     = np.random.binomial(1,sig(0.6*b[:,0]-0.5+0.02*(age-62)/15),N)
hypertension = np.random.binomial(1,sig(0.7*b[:,0]-0.2),N)
hf           = np.random.binomial(1,sig(0.4*b[:,1]+0.5*hypertension-1.5),N)
copd         = np.random.binomial(1,sig(0.5*b[:,2]-1.0),N)
ckd          = np.random.binomial(1,sig(0.5*b[:,0]+0.6*diabetes-1.2),N)
obesity      = np.random.binomial(1,sig(0.5*b[:,0]-0.8),N)
depression   = np.random.binomial(1,sig(0.3*b[:,3]-1.4),N)
n_comorb     = diabetes+hypertension+hf+copd+ckd
np.random.seed(21)
glucose      = np.random.normal(105+15*diabetes,28,N).clip(50,400).round(1)
creatinine   = np.random.gamma(1.6+0.8*hf,0.6,N).clip(0.4,10).round(2)
hba1c        = np.random.normal(6.5+0.8*diabetes,1.4,N).clip(4,14).round(1)
sbp          = np.random.normal(128+12*hypertension,18,N).clip(80,200).round(0)
bmi          = np.random.normal(28+4*obesity,6,N).clip(15,55).round(1)
logit = (-3.2+0.6*hf+0.5*diabetes+0.5*ckd+0.3*copd+0.02*(age-62)/15
         +0.3*(admit_type=="Emergency").astype(int)+0.25*(payer=="Medicaid").astype(int)
         +0.2*(los_days>7).astype(int)+0.5*hf*diabetes+np.random.normal(0,0.25,N))
readmitted = np.random.binomial(1,sig(logit),N)
# Add admission dates for temporal split demonstration
start = pd.Timestamp("2019-01-01"); np.random.seed(5)
admit_dates = [start+pd.Timedelta(days=int(d))
               for d in np.random.randint(0,(pd.Timestamp("2023-12-31")-start).days,N)]

df = pd.DataFrame({
    "patient_id":[f"PT-{i:05d}" for i in range(1,N+1)],
    "age":age,"sex":sex,"payer":payer,"admit_type":admit_type,"los_days":los_days,
    "diabetes":diabetes,"hypertension":hypertension,"hf":hf,"copd":copd,
    "ckd":ckd,"obesity":obesity,"depression":depression,"n_comorb":n_comorb,
    "glucose":glucose,"creatinine":creatinine,"hba1c":hba1c,"sbp":sbp,"bmi":bmi,
    "readmitted":readmitted,"admit_date":pd.to_datetime(admit_dates),
})
df = df.sort_values("admit_date").reset_index(drop=True)
df["los_gt7"] = (df.los_days>7).astype(int)
NUMERIC   = ["age","los_days","n_comorb","glucose","creatinine","hba1c","sbp","bmi"]
BINARY    = ["diabetes","hypertension","hf","copd","ckd","obesity","depression","los_gt7"]
CATEGORIC = ["sex","payer","admit_type"]
FEATURES  = NUMERIC + BINARY + CATEGORIC
TARGET    = "readmitted"
print(f"Dataset: {df.shape} | Positive rate: {df[TARGET].mean()*100:.1f}%")
print(f"Date range: {df.admit_date.min().date()} → {df.admit_date.max().date()}")


## 2. Why Temporal Splits Matter in Healthcare

Random splitting of clinical data **leaks future information into training** when:
- The same patient appears multiple times (train on visit 1, test on visit 2 of the *same patient*)
- Care patterns change over time (COVID, policy changes, new guidelines)
- Models evaluated on temporal holdouts reflect real-world deployment performance

| Split type | Use when | Risk |
|---|---|---|
| Random stratified | Cross-sectional data, no time structure | Data leakage if temporal trends exist |
| Temporal (chronological) | Time-ordered admissions, EHR data | Better reflects deployment conditions |
| Patient-level | Same patient across visits | Prevents same-patient leakage |
| Geographic | Multi-site data | Tests transportability |


In [ ]:
# ── Temporal split: train 2019-2021, validate 2022, test 2023 ─────────────────
train_mask = df.admit_date.dt.year <= 2021
val_mask   = df.admit_date.dt.year == 2022
test_mask  = df.admit_date.dt.year == 2023

df_train = df[train_mask].copy()
df_val   = df[val_mask].copy()
df_test  = df[test_mask].copy()

print("Temporal split summary:")
for name, sub in [("Train (≤2021)",df_train),("Val (2022)",df_val),("Test (2023)",df_test)]:
    print(f"  {name:16s}: N={len(sub):5d} | Positive rate={sub[TARGET].mean()*100:.1f}%")

# ── Also create a random stratified split for comparison ─────────────────────
X_all = df[FEATURES]; y_all = df[TARGET]
X_tr_rand, X_te_rand, y_tr_rand, y_te_rand = train_test_split(
    X_all, y_all, test_size=0.20, random_state=42, stratify=y_all)

# Visualise temporal vs random
fig, axes = plt.subplots(1,2,figsize=(13,4))
monthly = (df.set_index("admit_date").resample("ME")[TARGET].agg(["mean","count"]))
monthly["rate"] = monthly["mean"]*100

ax = axes[0]
colors_t = ["#4878CF" if y<=2021 else "#6ACC65" if y==2022 else "#D65F5F"
            for y in monthly.index.year]
ax.bar(monthly.index, monthly["rate"], color=colors_t, width=20, alpha=0.85)
from matplotlib.patches import Patch
legend_els = [Patch(color="#4878CF",label="Train (≤2021)"),
              Patch(color="#6ACC65",label="Val (2022)"),
              Patch(color="#D65F5F",label="Test (2023)")]
ax.legend(handles=legend_els, fontsize=9)
ax.set_ylabel("Readmission rate (%)"); ax.set_xlabel("Month")
ax.set_title("Temporal split — monthly readmission rate")
import matplotlib.dates as mdates
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
plt.setp(ax.xaxis.get_majorticklabels(),rotation=30,ha="right")

ax = axes[1]
for split,label,color in [("Temporal","Temporal split (2023)","#D65F5F"),
                            ("Random","Random split (20%)","#4878CF")]:
    if split=="Temporal":
        pos_rate = df_test[TARGET].mean()*100
        neg_rate = 100-pos_rate
    else:
        pos_rate = y_te_rand.mean()*100
        neg_rate = 100-pos_rate
    ax.bar(label,[neg_rate,pos_rate],bottom=[0,neg_rate],color=["#B0C4DE",color],
           alpha=0.8,edgecolor="white")
ax.set_ylabel("% of test set"); ax.set_title("Positive rate in test set by split strategy")
plt.tight_layout(); plt.savefig("/tmp/mod04/split_strategy.png",bbox_inches="tight"); plt.show()


## 3. Preprocessing Pipeline

In [ ]:
# sklearn Pipeline ensures: (1) no data leakage (fit on train, transform test)
#                         (2) reproducible, deployable preprocessing

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])
binary_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
])
categoric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe",     OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num",  numeric_transformer,   NUMERIC),
    ("bin",  binary_transformer,    BINARY),
    ("cat",  categoric_transformer, CATEGORIC),
])

# Fit on train, transform val and test
X_train = df_train[FEATURES]; y_train = df_train[TARGET]
X_val   = df_val[FEATURES];   y_val   = df_val[TARGET]
X_test  = df_test[FEATURES];  y_test  = df_test[TARGET]

preprocessor.fit(X_train)
X_train_pp = preprocessor.transform(X_train)
X_val_pp   = preprocessor.transform(X_val)
X_test_pp  = preprocessor.transform(X_test)

print(f"Preprocessed shapes:")
print(f"  Train : {X_train_pp.shape}")
print(f"  Val   : {X_val_pp.shape}")
print(f"  Test  : {X_test_pp.shape}")
print()

# Feature names after OHE
ohe_names = preprocessor.named_transformers_["cat"]["ohe"].get_feature_names_out(CATEGORIC)
all_feature_names = NUMERIC + BINARY + list(ohe_names)
print(f"Total features after encoding: {len(all_feature_names)}")


## 4. Class Imbalance — Diagnosis & Remedies

In [ ]:
# ── Imbalance check ───────────────────────────────────────────────────────────
pos_rate = y_train.mean()
print(f"Class distribution in train set:")
print(f"  Positives (readmitted)    : {y_train.sum():5d} ({pos_rate*100:.1f}%)")
print(f"  Negatives (not readmitted): {(y_train==0).sum():5d} ({(1-pos_rate)*100:.1f}%)")
print(f"  Imbalance ratio           : 1 : {(1-pos_rate)/pos_rate:.1f}")

# Strategy comparison
from collections import Counter
try:
    from imblearn.over_sampling  import SMOTE
    from imblearn.under_sampling import RandomUnderSampler
    from imblearn.combine        import SMOTETomek
    imblearn_ok = True
except ImportError:
    imblearn_ok = False
    print("\nInstall imbalanced-learn: pip install imbalanced-learn")

strategies = {"Original": (X_train_pp, y_train.values)}
if imblearn_ok:
    sm  = SMOTE(random_state=42)
    rus = RandomUnderSampler(random_state=42)
    smt = SMOTETomek(random_state=42)
    for name, sampler in [("SMOTE",sm),("Under-sample",rus),("SMOTE+Tomek",smt)]:
        X_res, y_res = sampler.fit_resample(X_train_pp, y_train.values)
        strategies[name] = (X_res, y_res)

fig, ax = plt.subplots(figsize=(9,4))
names, pos_counts, neg_counts = [], [], []
for name,(Xr,yr) in strategies.items():
    c = Counter(yr)
    names.append(name); pos_counts.append(c[1]); neg_counts.append(c[0])
x = np.arange(len(names)); w = 0.35
ax.bar(x-w/2, neg_counts, w, label="Negative", color="#4878CF", alpha=0.85)
ax.bar(x+w/2, pos_counts, w, label="Positive", color="#D65F5F", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=10)
ax.set_ylabel("N samples"); ax.set_title("Class distribution by resampling strategy")
ax.legend(fontsize=9)
for i,(n,p) in enumerate(zip(neg_counts,pos_counts)):
    ax.text(i-w/2,n+20,str(n),ha="center",fontsize=8)
    ax.text(i+w/2,p+20,str(p),ha="center",fontsize=8)
plt.tight_layout(); plt.savefig("/tmp/mod04/class_imbalance.png",bbox_inches="tight"); plt.show()


## 5. Baseline Models — the Bar to Beat

In [ ]:
# Every ML project needs baselines. If your fancy model can't beat these, something is wrong.
baselines = {
    "Majority class (always predict 0)": DummyClassifier(strategy="most_frequent"),
    "Random (stratified)":               DummyClassifier(strategy="stratified",random_state=42),
    "Logistic Regression (L2)":          LogisticRegression(C=1.0,max_iter=1000,
                                                             class_weight="balanced",random_state=42),
}

results = []
for name, model in baselines.items():
    model.fit(X_train_pp, y_train)
    y_pred     = model.predict(X_val_pp)
    y_prob     = model.predict_proba(X_val_pp)[:,1]
    auc        = roc_auc_score(y_val, y_prob)
    ap         = average_precision_score(y_val, y_prob)
    brier      = brier_score_loss(y_val, y_prob)
    results.append({"Model":name,"AUC":round(auc,4),"Avg Precision":round(ap,4),
                    "Brier":round(brier,4)})
    print(f"{name}")
    print(f"  AUC={auc:.4f} | Avg Precision={ap:.4f} | Brier={brier:.4f}")
    print()

baseline_df = pd.DataFrame(results)
print("\nBaseline comparison:")
display(baseline_df)
print("\n→ Module 04 goal: beat Logistic Regression AUC on the 2023 test set.")


## 6. Cross-Validation Strategies

In [ ]:
# Compare stratified k-fold vs time-series CV on the training set
lr = LogisticRegression(C=1.0,max_iter=1000,class_weight="balanced",random_state=42)

# Stratified k-fold (appropriate for cross-sectional data)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_skf = cross_val_score(lr, X_train_pp, y_train, cv=skf, scoring="roc_auc")

# Time-series split (for temporal data — each fold extends the train window)
tss = TimeSeriesSplit(n_splits=5)
cv_tss = cross_val_score(lr, X_train_pp, y_train, cv=tss, scoring="roc_auc")

fig, axes = plt.subplots(1,2,figsize=(13,4))
for ax, scores, title, color in [
    (axes[0], cv_skf, "Stratified 5-Fold CV (shuffled — ignores time order)", "#4878CF"),
    (axes[1], cv_tss, "Time-Series Split CV (expanding window — respects time order)", "#D65F5F"),
]:
    ax.bar(range(1,6), scores, color=color, alpha=0.85, edgecolor="white")
    ax.axhline(scores.mean(), color="black", ls="--", lw=1.5,
               label=f"Mean AUC = {scores.mean():.4f} ± {scores.std():.4f}")
    ax.set_xlabel("Fold"); ax.set_ylabel("AUC-ROC")
    ax.set_title(title); ax.legend(fontsize=9); ax.set_ylim(0.5,0.9)
    for i,v in enumerate(scores):
        ax.text(i+1,v+0.003,f"{v:.3f}",ha="center",fontsize=8)

plt.tight_layout(); plt.savefig("/tmp/mod04/cv_comparison.png",bbox_inches="tight"); plt.show()
print(f"Stratified CV : {cv_skf.mean():.4f} ± {cv_skf.std():.4f}")
print(f"Time-series CV: {cv_tss.mean():.4f} ± {cv_tss.std():.4f}")
print()
print("Use Time-Series CV when your data is temporally ordered and you care about")
print("prospective performance (real deployment conditions).")


## 7. Knowledge Check
1. You have 5 years of EHR data. A colleague proposes random 80/20 split. What are the risks, and what would you recommend instead?
2. Your positive class rate is 13%. What class-weight setting should you pass to sklearn models?
3. SMOTE creates synthetic samples. What is the risk of applying SMOTE *before* splitting into train/test?
4. Logistic regression baseline AUC = 0.74. Your XGBoost gets 0.75. Is this improvement clinically meaningful?
5. Implement patient-level split: ensure no patient appears in both train and test.


In [ ]:
# Exercise 5 — patient-level split (each PT only in one split)
# In real datasets, use MRN/patient_id grouping
from sklearn.model_selection import GroupShuffleSplit
groups = np.array([int(pid.split("-")[1]) % 100 for pid in df["patient_id"]])  # simulate patient groups
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
tr_idx, te_idx = next(gss.split(df[FEATURES], df[TARGET], groups=groups))
print(f"Patient-level split: train N={len(tr_idx)}, test N={len(te_idx)}")
# Verify no patient group overlap
train_groups = set(groups[tr_idx]); test_groups = set(groups[te_idx])
overlap = train_groups & test_groups
print(f"Patient group overlap: {len(overlap)} groups (should be 0 for true patient-level split)")


***
**Next → NB-02: Decision Trees & Random Forest**
